# Phase D MFA-TrimCI: Reproduce Best D2 Result

This notebook runs the Phase D MFA-TrimCI workflow from scratch and displays the outputs directly in the notebook.

It regenerates:

- D1 overlapping determinant-count check.
- D2 h1diag baseline at `threshold=0.06`.
- D2 h1diag best run at `threshold=0.01`, `max_final_dets=1000`.
- D2 balanced partition ablation at `threshold=0.06`.

The headline result is the D2 h1diag best run:

```text
E_total ≈ -326.568151 Ha
error vs -327.1920 Ha ≈ +0.623849 Ha
fragment_n_dets = [1008, 1008, 1]
total_dets = 2017
```

Expected runtime: roughly 1-2 minutes, depending on local load.

## Setup

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display

# Make imports robust whether the notebook is launched from Proj_Flow/ or TrimCI_Flow/.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "TrimCI_Flow":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from TrimCI_Flow.mfa import run_mfa_d1, run_mfa_d2

DATA_DIR = PROJECT_ROOT / "Fe4S4_251230orbital_-327.1920_10kdets" / "Fe4S4_251230orbital_-327.1920_10kdets"
FCIDUMP = DATA_DIR / "fcidump_cycle_6"
REF_DETS = DATA_DIR / "dets.npz"
GAMMA = PROJECT_ROOT / "TrimCI_Flow" / "Outputs" / "meanfield_active" / "outs_extraction_autodets" / "gamma_mixed_final.npy"
MFA_OUT = PROJECT_ROOT / "TrimCI_Flow" / "Outputs" / "mfa"
REFERENCE_ENERGY = -327.1920

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"FCIDUMP      = {FCIDUMP}")
print(f"REF_DETS     = {REF_DETS}")
print(f"GAMMA        = {GAMMA}")
print(f"RUN_STAMP    = {RUN_STAMP}")

for path in (FCIDUMP, REF_DETS, GAMMA):
    if not path.exists():
        raise FileNotFoundError(path)

PROJECT_ROOT = /home/unfunnypanda/Proj_Flow
FCIDUMP      = /home/unfunnypanda/Proj_Flow/Fe4S4_251230orbital_-327.1920_10kdets/Fe4S4_251230orbital_-327.1920_10kdets/fcidump_cycle_6
REF_DETS     = /home/unfunnypanda/Proj_Flow/Fe4S4_251230orbital_-327.1920_10kdets/Fe4S4_251230orbital_-327.1920_10kdets/dets.npz
GAMMA        = /home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/meanfield_active/outs_extraction_autodets/gamma_mixed_final.npy
RUN_STAMP    = 20260417_141355


In [2]:
def trimci_config(threshold, max_final_dets):
    return {
        "threshold": threshold,
        "max_final_dets": max_final_dets,
        "max_rounds": 2,
        "num_runs": 1,
        "pool_build_strategy": "heat_bath",
        "verbose": False,
    }


CONFIG_D1 = trimci_config(0.06, "auto")
CONFIG_D2_BASELINE = trimci_config(0.06, "auto")
CONFIG_D2_BEST = trimci_config(0.01, 1000)
CONFIG_D2_BALANCED = trimci_config(0.06, "auto")

display(Markdown("### TrimCI configurations"))
print("D1:", CONFIG_D1)
print("D2 baseline:", CONFIG_D2_BASELINE)
print("D2 best:", CONFIG_D2_BEST)
print("D2 balanced:", CONFIG_D2_BALANCED)

### TrimCI configurations

D1: {'threshold': 0.06, 'max_final_dets': 'auto', 'max_rounds': 2, 'num_runs': 1, 'pool_build_strategy': 'heat_bath', 'verbose': False}
D2 baseline: {'threshold': 0.06, 'max_final_dets': 'auto', 'max_rounds': 2, 'num_runs': 1, 'pool_build_strategy': 'heat_bath', 'verbose': False}
D2 best: {'threshold': 0.01, 'max_final_dets': 1000, 'max_rounds': 2, 'num_runs': 1, 'pool_build_strategy': 'heat_bath', 'verbose': False}
D2 balanced: {'threshold': 0.06, 'max_final_dets': 'auto', 'max_rounds': 2, 'num_runs': 1, 'pool_build_strategy': 'heat_bath', 'verbose': False}


## D1: Overlapping Determinant-Count Check

This reproduces the Phase C/B-compatible overlapping result. D1 does not report a total energy because fragments overlap.

In [3]:
d1_out = MFA_OUT / f"outs_notebook_d1_overlapping_{RUN_STAMP}"

d1 = run_mfa_d1(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D1,
    output_dir=str(d1_out),
)

display(Markdown(f"""
### D1 Result

- Output: `{d1_out}`
- fragment_n_dets: `{d1['fragment_n_dets']}`
- total_dets: `{d1['total_dets']}`
- matches Phase C baseline: `{d1['matches_phase_c_baseline']}`
- energy_status: `{d1['energy_status']}`
"""))

[MFA-D1] n_orb=36, n_elec=54, E_nuc=0.0
[MFA-D1] gamma_mixed: shape=(36, 36), Tr=54.0000
[MFA-D1 WARNING] gamma_source loaded as diagonal_vector_promoted_to_matrix
[MFA-D1] 3 overlapping fragments (W=15, S=10)
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 32207175
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 105 entries in 0.003131s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.06, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.054, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0486, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.04374, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.039366, restarting from initial pool=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 120, final threshold: 0.039366
[Trim] Start. Pool=120
[Trim] Core-core H precomputed: 1


### D1 Result

- Output: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d1_overlapping_20260417_141355`
- fragment_n_dets: `[51, 51, 16]`
- total_dets: `118`
- matches Phase C baseline: `True`
- energy_status: `diagnostic_only_overlapping_no_total`


## D2 Baseline: h1diag Partition, threshold=0.06

This is the quick non-overlapping energy baseline.

In [4]:
d2_baseline_out = MFA_OUT / f"outs_notebook_d2_h1diag_threshold006_{RUN_STAMP}"

d2_baseline = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BASELINE,
    output_dir=str(d2_baseline_out),
    partition="h1diag",
)

display(Markdown(f"""
### D2 Baseline Result

- Output: `{d2_baseline_out}`
- E_total: `{d2_baseline['E_total']:.12f} Ha`
- error vs reference: `{d2_baseline['error_vs_reference']:+.12f} Ha`
- fragment_n_dets: `{d2_baseline['fragment_n_dets']}`
- total_dets: `{d2_baseline['total_dets']}`
- gamma_load_mode: `{d2_baseline['gamma_load_mode']}`
"""))

[MFA-D2] n_orb=36, n_elec=54, E_nuc=0.0
[MFA-D2] gamma_mixed: shape=(36, 36), Tr=54.0000
[MFA-D2 WARNING] gamma_source loaded as diagonal_vector_promoted_to_matrix
[MFA-D2] 3 non-overlapping fragments of 12 orbitals each (partition=h1diag)
[MFA-D2] E_mf_global = -323.080403 Ha (E_nuc=0.0)
[MFA-D2] Row-partition sum=-323.080403, E_mf_global_elec=-323.080403, match=True
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 853776
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 66 entries in 0.002093s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.0600000000, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0540000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0486000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0437400000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0393660000, re


### D2 Baseline Result

- Output: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_h1diag_threshold006_20260417_141355`
- E_total: `-326.448505882566 Ha`
- error vs reference: `+0.743494117434 Ha`
- fragment_n_dets: `[73, 73, 1]`
- total_dets: `147`
- gamma_load_mode: `diagonal_vector_promoted_to_matrix`


## D2 Best Run: h1diag Partition, threshold=0.01, max_final_dets=1000

This is the current best Phase D v1 result. This cell is the most expensive one.

In [5]:
d2_best_out = MFA_OUT / f"outs_notebook_d2_h1diag_threshold001_1000dets_{RUN_STAMP}"

d2_best = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BEST,
    output_dir=str(d2_best_out),
    partition="h1diag",
)

display(Markdown(f"""
### D2 Best Result

- Output: `{d2_best_out}`
- **E_total: `{d2_best['E_total']:.12f} Ha`**
- **error vs reference: `{d2_best['error_vs_reference']:+.12f} Ha`**
- fragment_n_dets: `{d2_best['fragment_n_dets']}`
- total_dets: `{d2_best['total_dets']}`
- E_total - DMET 1-shot B: `{d2_best['E_total_minus_dmet_1shot_b']:+.12f} Ha`
- gamma_load_mode: `{d2_best['gamma_load_mode']}`
"""))

[MFA-D2] n_orb=36, n_elec=54, E_nuc=0.0
[MFA-D2] gamma_mixed: shape=(36, 36), Tr=54.0000
[MFA-D2 WARNING] gamma_source loaded as diagonal_vector_promoted_to_matrix
[MFA-D2] 3 non-overlapping fragments of 12 orbitals each (partition=h1diag)
[MFA-D2] E_mf_global = -323.080403 Ha (E_nuc=0.0)
[MFA-D2] Row-partition sum=-323.080403, E_mf_global_elec=-323.080403, match=True
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 853776
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 66 entries in 0.002144s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.0100000000, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 620, final threshold: 0.0100000000
[Trim] Start. Pool=620
[Trim] Core-core H precomputed: 1 dets, 1 nnz, 0.0161719590s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=11
[Trim] Final diagonalization...
[Trim


### D2 Best Result

- Output: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_h1diag_threshold001_1000dets_20260417_141355`
- **E_total: `-326.566604075483 Ha`**
- **error vs reference: `+0.625395924517 Ha`**
- fragment_n_dets: `[1008, 1008, 1]`
- total_dets: `2017`
- E_total - DMET 1-shot B: `-67.609542075483 Ha`
- gamma_load_mode: `diagonal_vector_promoted_to_matrix`


## D2 Balanced Partition Ablation

This checks whether removing the one-determinant closed fragment improves the energy. It should produce nontrivial determinant counts on all fragments, but the earlier ablation showed it is energetically worse than h1diag for diagonal-density MFA.

In [6]:
d2_balanced_out = MFA_OUT / f"outs_notebook_d2_balanced_threshold006_{RUN_STAMP}"

d2_balanced = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BALANCED,
    output_dir=str(d2_balanced_out),
    partition="balanced",
)

display(Markdown(f"""
### D2 Balanced Result

- Output: `{d2_balanced_out}`
- E_total: `{d2_balanced['E_total']:.12f} Ha`
- error vs reference: `{d2_balanced['error_vs_reference']:+.12f} Ha`
- fragment_n_dets: `{d2_balanced['fragment_n_dets']}`
- total_dets: `{d2_balanced['total_dets']}`
- fragment_n_alpha: `{d2_balanced['fragment_n_alpha']}`
- fragment_n_beta: `{d2_balanced['fragment_n_beta']}`
"""))

[MFA-D2] n_orb=36, n_elec=54, E_nuc=0.0
[MFA-D2] gamma_mixed: shape=(36, 36), Tr=54.0000
[MFA-D2 WARNING] gamma_source loaded as diagonal_vector_promoted_to_matrix
[MFA-D2] 3 non-overlapping fragments of 12 orbitals each (partition=balanced)
[MFA-D2] E_mf_global = -323.080403 Ha (E_nuc=0.0)
[MFA-D2] Row-partition sum=-323.080403, E_mf_global_elec=-323.080403, match=True
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 14520
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 66 entries in 0.001497s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.0600000000, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0540000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0486000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0437400000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0393660000, r


### D2 Balanced Result

- Output: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_balanced_threshold006_20260417_141355`
- E_total: `-326.405370500702 Ha`
- error vs reference: `+0.786629499298 Ha`
- fragment_n_dets: `[73, 73, 73]`
- total_dets: `219`
- fragment_n_alpha: `[9, 9, 9]`
- fragment_n_beta: `[10, 9, 8]`


## Summary Table

In [7]:
runs = [
    ("D1 overlapping", d1),
    ("D2 h1diag 0.06", d2_baseline),
    ("D2 h1diag 0.01 / 1000 dets", d2_best),
    ("D2 balanced 0.06", d2_balanced),
]

rows = []
for label, result in runs:
    rows.append({
        "run": label,
        "partition": result.get("partition"),
        "E_total": result.get("E_total"),
        "error_vs_reference": result.get("error_vs_reference"),
        "fragment_n_dets": result.get("fragment_n_dets"),
        "total_dets": result.get("total_dets"),
        "gamma_load_mode": result.get("gamma_load_mode"),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except Exception:
    for row in rows:
        print(row)

best_energy = d2_best["E_total"]
best_error = d2_best["error_vs_reference"]
display(Markdown(f"""
### Headline

The best regenerated D2 result in this notebook is:

```text
E_total = {best_energy:.12f} Ha
error vs {REFERENCE_ENERGY:.4f} Ha = {best_error:+.12f} Ha
fragment_n_dets = {d2_best['fragment_n_dets']}
total_dets = {d2_best['total_dets']}
```
"""))

{'run': 'D1 overlapping', 'partition': 'overlapping_W15_S10', 'E_total': None, 'error_vs_reference': None, 'fragment_n_dets': [51, 51, 16], 'total_dets': 118, 'gamma_load_mode': 'diagonal_vector_promoted_to_matrix'}
{'run': 'D2 h1diag 0.06', 'partition': 'nonoverlapping_12_12_12_h1diag', 'E_total': -326.448505882566, 'error_vs_reference': 0.7434941174340111, 'fragment_n_dets': [73, 73, 1], 'total_dets': 147, 'gamma_load_mode': 'diagonal_vector_promoted_to_matrix'}
{'run': 'D2 h1diag 0.01 / 1000 dets', 'partition': 'nonoverlapping_12_12_12_h1diag', 'E_total': -326.56660407548264, 'error_vs_reference': 0.6253959245173633, 'fragment_n_dets': [1008, 1008, 1], 'total_dets': 2017, 'gamma_load_mode': 'diagonal_vector_promoted_to_matrix'}
{'run': 'D2 balanced 0.06', 'partition': 'nonoverlapping_12_12_12_balanced_refdet_roundrobin', 'E_total': -326.40537050070174, 'error_vs_reference': 0.7866294992982716, 'fragment_n_dets': [73, 73, 73], 'total_dets': 219, 'gamma_load_mode': 'diagonal_vector_pr


### Headline

The best regenerated D2 result in this notebook is:

```text
E_total = -326.566604075483 Ha
error vs -327.1920 Ha = +0.625395924517 Ha
fragment_n_dets = [1008, 1008, 1]
total_dets = 2017
```


## Interpretation

The notebook should reproduce the Phase D v1 conclusion:

- D1 preserves the Phase C/B determinant baseline.
- D2 h1diag gives the best non-overlapping energy result.
- Tightening TrimCI to `threshold=0.01`, `max_final_dets=1000` improves energy modestly but does not close the remaining gap.
- Balanced partition removes the closed-fragment determinant pathology but is energetically worse for the current diagonal-density MFA model.

The current best Phase D result is therefore the h1diag D2 run at `threshold=0.01`, `max_final_dets=1000`.

In [8]:
display(Markdown(f"""
### Output folders created by this notebook

- D1: `{d1_out}`
- D2 baseline: `{d2_baseline_out}`
- D2 best: `{d2_best_out}`
- D2 balanced: `{d2_balanced_out}`
"""))


### Output folders created by this notebook

- D1: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d1_overlapping_20260417_141355`
- D2 baseline: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_h1diag_threshold006_20260417_141355`
- D2 best: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_h1diag_threshold001_1000dets_20260417_141355`
- D2 balanced: `/home/unfunnypanda/Proj_Flow/TrimCI_Flow/Outputs/mfa/outs_notebook_d2_balanced_threshold006_20260417_141355`
